# GEE unfolding (gee) (`unfold_gee`)

This notebook applies **GEE unfolding with robust Liang-Zeger sandwich inference** — the Python analogue of the R
package **gee** — to a realistic benchmark: detector readings
synthesised from the **Monte-Carlo calculated spectrum
`t4-14-s.txt_1`** of the
[IAEA Compendium](https://www-nds.iaea.org/benchmarks/), a BNCT-like
beam-shaping-assembly spectrum with a thermal group, an epithermal
$1/E$ region and a fast peak.

We use the built-in GSF response functions (10 Bonner spheres, `0in` –
`18in`, 60 energy bins from 1e-9 to ~631 MeV).  Detector readings are
folded with `Detector.get_effective_readings_for_spectra`, the
spectrum is reconstructed with `unfold_gee`, and the result is
compared against the ground truth — which never enters the unfolding.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bssunfold import Detector, RF_GSF
from bssunfold.utils.comparison import compare_spectra

detector = Detector(RF_GSF)
E = detector.E_MeV
names = detector.detector_names
print(f"Detector grid: {detector.n_energy_bins} bins, "
      f"{E[0]:.1e} - {E[-1]:.1f} MeV")
print("Spheres:", ", ".join(names))
detector.plot_response_functions()


## 1. IAEA Compendium reference spectrum → detector readings

The compendium CSV stores 61-point Monte-Carlo spectra on its
own energy grid; `get_effective_readings_for_spectra` folds
the spectrum with the response functions and resamples it
onto the 60-bin detector grid.

In [ ]:
reference_csv = pd.read_csv(
    '../tests/MonteCarlo_Calculated_spectra_from_IAEA_Comp_for_comparison.csv'
)
readings = detector.get_effective_readings_for_spectra(
    reference_csv[['E_MeV', 't4-14-s.txt_1']]
)
print("Effective readings:")
for nm in names:
    print(f"  {nm:>5s}: {readings[nm]:.4g}")

phi_true = np.interp(
    E, reference_csv['E_MeV'].values, reference_csv['t4-14-s.txt_1'].values
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.loglog(E, phi_true, "k-", lw=1.5)
ax.set(xlabel="E, MeV", ylabel=r"$\varphi(E)$, cm$^{-2}$s$^{-1}$bin$^{-1}$",
       title="IAEA Compendium spectrum t4-14-s.txt_1 (ground truth)")
ax.grid(True, which="both", ls=":", alpha=0.5)

ax = axes[1]
vals = [readings[nm] for nm in names]
ax.bar(np.arange(len(names)), vals, color="steelblue")
ax.set_yscale("log")
ax.set_xticks(np.arange(len(names)))
ax.set_xticklabels(names, rotation=45)
ax.set(xlabel="sphere", ylabel="reading, a.u.",
       title="Effective Bonner-sphere readings")
ax.grid(True, axis="y", ls=":", alpha=0.5)
fig.tight_layout()
plt.show()


## 2. GEE unfolding with robust inference

`unfold_gee` (R `gee` 4.13-30 analogue, Liang & Zeger 1986) treats
the spheres as a correlated cluster: the IRLS loop solves the
penalised score `A^T R(alpha)^-1 (b-Ax) - lam G x = 0` and reports
the robust Liang-Zeger **sandwich** standard errors of the
spectrum — uncertainties that stay consistent when the working
correlation is misspecified.

In [ ]:
result = detector.unfold_gee(readings, save_result=False)

print(f"method        : {result['method']}")
print(f"family/corstr : {result['family']}/{result['corstr']}")
print(f"alpha         : {result['alpha']:.3f}  (dispersion "
      f"phi = {result['phi']:.3g})")
print(f"pearson chi2  : {result['pearson_chi2']:.3g}")
print(f"converged     : {result['gee_converged']} "
      f"({result['iterations']} iterations)")

robust_se = result['robust_se']
print("robust SE on the first bins:", np.round(robust_se[:5], 2))

lines_to_plot = [
    ("GEE (gaussian / exchangeable)", result['spectrum'], "C1-"),
]


## 3. Working-correlation comparison and robust SE coverage

Compare the three working-correlation structures, then check the
robust sandwich uncertainties against a small Monte-Carlo run
over Poisson-noised readings (the SEs should bracket the spread of
unfolding outcomes).

In [ ]:
results_cor = {}
for corstr in ["independence", "exchangeable", "ar1"]:
    res = detector.unfold_gee(readings, corstr=corstr, save_result=False)
    results_cor[corstr] = res
    q = compare_spectra(
        res['spectrum'], phi_true,
        metrics=['pearson_r', 'relative_flux_error',
                 'comprehensive_score'],
    )
    print(f"{corstr:>13s}: alpha={res['alpha']:+.3f}  "
          f"pearson_r={q['pearson_r']:.3f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(E, phi_true, "k-", lw=2, label="IAEA ground truth")
for corstr, res in results_cor.items():
    ax.loglog(E, res['spectrum'], lw=1.1, label=corstr)
ax.set(xlabel="E, MeV", ylabel=r"$\varphi(E)$, cm$^{-2}$s$^{-1}$bin$^{-1}$",
       title="GEE unfolding: working correlation structures")
ax.set_xlim(E[0], 30)
ax.grid(True, which="both", ls=":", alpha=0.35)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

# Monte-Carlo sanity check of the robust SE (should be ~1 sigma-ish)
rng = np.random.default_rng(0)
n_mc = 20
mc_spectra = []
for _ in range(n_mc):
    noisy = np.maximum(
        readings_scaled := np.array([readings[n] for n in names]) * 1e6,
        0.0,
    )
    noisy = rng.poisson(noisy) / 1e6
    rc = {n: float(v) for n, v in zip(names, noisy)}
    mc_spectra.append(detector.unfold_gee(rc, save_result=False)['spectrum'])
mc_arr = np.array(mc_spectra)
mc_std = mc_arr.std(axis=0, ddof=1)
mask = robust_se > 0
ratio = mc_std[mask] / robust_se[mask]
print(f"MC spread / robust SE, median = {np.median(ratio):.2f} "
      f"(robust sandwich should be of the MC-noise order)")


## Quality assessment

`compare_spectra` reports the reconstruction metrics against the
independently known IAEA Compendium spectrum (used only for
evaluation).

In [ ]:
quality = compare_spectra(
    result['spectrum'], phi_true,
    metrics=["relative_flux_error", "pearson_r", "comprehensive_score",
             "fluence_difference_percent", "dose_difference_percent"],
    energy=E,
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(E, phi_true, "k-", lw=2, label="IAEA ground truth")
for label, spec, style in lines_to_plot:
    ax.loglog(E, spec, style, lw=1.2, label=label)
ax.set(xlabel="E, MeV", ylabel=r"$\varphi(E)$, cm$^{-2}$s$^{-1}$bin$^{-1}$",
       title="GEE (robust sandwich) unfolding")
ax.set_xlim(E[0], 30)
ax.grid(True, which="both", ls=":", alpha=0.35)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

for k, v in quality.items():
    print(f"{k:>26s}: {v}")
